# Candidate K and user-history sparsity

## tl;dr

Keep `candidate_k=100` only as a retrieval budget and recall-ceiling diagnostic. Use Recall@10/20/50 for candidate quality and Recall/NDCG/MRR@5/10 for the user-facing ranking. The benchmark covers only users with at least three distinct restaurant interactions, so it represents 34.3% of observed users.

## Context & Methods

Source: immutable artifacts from `artifacts/runs/20260921T084455858595Z-03a76325`. User counts use the deduplicated modeling interaction grain. Candidate recall uses relevant test queries only (`relevance > 0`) and the RRF candidate rank.

### Key assumptions

- K=100 is interpreted as an internal retrieval budget, not a displayed recommendation list.
- The held-out restaurant is the next observed review interaction, not necessarily the user's complete real-world next visit.
- Users with fewer than three distinct restaurant interactions are excluded from this seen-user benchmark.

In [1]:
import json
import os
from collections import Counter
from pathlib import Path
from statistics import mean, median

os.environ.setdefault('MPLCONFIGDIR', 'artifacts/.matplotlib')
import matplotlib.pyplot as plt
import numpy as np

RUN_DIR = Path('artifacts/runs/20260921T084455858595Z-03a76325')
OUTPUT_DIR = Path('analysis')
OUTPUT_DIR.mkdir(exist_ok=True)

def read_jsonl(path):
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle]


## Data

First inspect the full user population and the benchmark-eligible population.

In [2]:
dataset = read_jsonl(RUN_DIR / 'dataset.jsonl')
user_counts = Counter(row['user_id'] for row in dataset)
eligible_counts = np.array([count for count in user_counts.values() if count >= 3])
history_distribution = {
    '1': sum(count == 1 for count in user_counts.values()),
    '2': sum(count == 2 for count in user_counts.values()),
    '3': sum(count == 3 for count in user_counts.values()),
    '4': sum(count == 4 for count in user_counts.values()),
    '5+': sum(count >= 5 for count in user_counts.values()),
}
population_summary = {
    'users': len(user_counts),
    'eligible_users_3plus': len(eligible_counts),
    'eligible_share': len(eligible_counts) / len(user_counts),
    'five_plus_all_user_share': history_distribution['5+'] / len(user_counts),
    'five_plus_eligible_share': history_distribution['5+'] / len(eligible_counts),
    'eligible_median_interactions': float(np.median(eligible_counts)),
    'eligible_p90_interactions': float(np.quantile(eligible_counts, 0.90)),
    'eligible_max_interactions': int(eligible_counts.max()),
}
print(json.dumps({'history_distribution': history_distribution, **population_summary}, indent=2))

fig, axis = plt.subplots(figsize=(8, 4.5))
bars = axis.bar(history_distribution.keys(), history_distribution.values(), color='#3568A8')
axis.set_title('Users by number of distinct restaurant interactions')
axis.set_xlabel('Interactions per user')
axis.set_ylabel('Users')
axis.grid(axis='y', color='#D9DEE7', linewidth=0.8)
axis.set_axisbelow(True)
axis.bar_label(bars, padding=3, fmt='%d')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'candidate_k_history_distribution.png', dpi=160)
plt.close(fig)


{
  "history_distribution": {
    "1": 3461,
    "2": 1130,
    "3": 650,
    "4": 399,
    "5+": 1347
  },
  "users": 6987,
  "eligible_users_3plus": 2396,
  "eligible_share": 0.34292257048804925,
  "five_plus_all_user_share": 0.19278660369257192,
  "five_plus_eligible_share": 0.5621869782971619,
  "eligible_median_interactions": 5.0,
  "eligible_p90_interactions": 14.0,
  "eligible_max_interactions": 93
}


## Results

Compute RRF candidate recall at additional cutoffs and compare the final LambdaMART output at K=5 and K=10.

In [3]:
queries = {
    row['query_id']: row
    for row in read_jsonl(RUN_DIR / 'queries.jsonl')
    if row['phase'] == 'test'
}
candidate_sizes = Counter()
target_ranks = {}
with (RUN_DIR / 'candidates_test.jsonl').open(encoding='utf-8') as handle:
    for line in handle:
        row = json.loads(line)
        query = queries[row['query_id']]
        candidate_sizes[row['query_id']] += 1
        if query['relevance'] > 0 and row['restaurant_id'] == query['target_restaurant_id']:
            target_ranks[row['query_id']] = int(row['candidate_rank'])

cutoffs = [5, 10, 20, 50, 100]
relevant_queries = [query for query in queries.values() if query['relevance'] > 0]

def recall_at(query_rows, cutoff):
    return sum(target_ranks.get(query['query_id'], 10**9) <= cutoff for query in query_rows) / len(query_rows)

candidate_recall = {cutoff: recall_at(relevant_queries, cutoff) for cutoff in cutoffs}
recall_by_history = {}
for bucket in ('history_1_2', 'history_3_plus'):
    bucket_queries = [query for query in relevant_queries if query['history_depth'] == bucket]
    recall_by_history[bucket] = {
        'queries': len(bucket_queries),
        **{cutoff: recall_at(bucket_queries, cutoff) for cutoff in cutoffs},
    }

metrics = json.loads((RUN_DIR / 'metrics.json').read_text(encoding='utf-8'))
ranker_metrics = metrics['test'].get('r1_lambdarank', metrics['test'].get('e4_lambdarank'))
if ranker_metrics is None:
    raise KeyError('Expected r1_lambdarank or legacy e4_lambdarank metrics')
final_recall = {
    5: ranker_metrics['recall_at_5'],
    10: ranker_metrics['recall_at_10'],
}
result_summary = {
    'catalog_items': len({row['restaurant_id'] for row in dataset}),
    'k100_catalog_share': 100 / len({row['restaurant_id'] for row in dataset}),
    'candidate_count_mean': mean(candidate_sizes.values()),
    'candidate_count_median': median(candidate_sizes.values()),
    'candidate_recall': candidate_recall,
    'final_recall': final_recall,
    'recall_by_history': recall_by_history,
    'target_availability': metrics['test']['target_availability']['rate'],
}
print(json.dumps(result_summary, indent=2))

fig, axis = plt.subplots(figsize=(8, 4.8))
axis.plot(cutoffs, [candidate_recall[k] for k in cutoffs], marker='o', color='#3568A8', label='C3 RRF · all relevant queries')
axis.plot(cutoffs, [recall_by_history['history_1_2'][k] for k in cutoffs], marker='s', linestyle='--', color='#D69E2E', label='C3 RRF · history=2')
axis.plot(cutoffs, [recall_by_history['history_3_plus'][k] for k in cutoffs], marker='^', linestyle=':', color='#73845C', label='C3 RRF · history≥3')
axis.plot([5, 10], [final_recall[5], final_recall[10]], marker='D', color='#B4557C', label='R1 LambdaMART final')
axis.set_title('Test recall by cutoff and history depth')
axis.set_xlabel('K')
axis.set_ylabel('Recall')
axis.set_xticks(cutoffs)
axis.set_ylim(0, 0.52)
axis.grid(color='#D9DEE7', linewidth=0.8)
axis.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'candidate_k_recall_curve.png', dpi=160)
plt.close(fig)


{
  "catalog_items": 748,
  "k100_catalog_share": 0.13368983957219252,
  "candidate_count_mean": 99.53255425709516,
  "candidate_count_median": 100.0,
  "candidate_recall": {
    "5": 0.06851063829787234,
    "10": 0.10936170212765957,
    "20": 0.16893617021276597,
    "50": 0.31106382978723407,
    "100": 0.4672340425531915
  },
  "final_recall": {
    "5": 0.06936170212765957,
    "10": 0.11574468085106383
  },
  "recall_by_history": {
    "history_1_2": {
      "queries": 631,
      "5": 0.09033280507131537,
      "10": 0.12361331220285261,
      "20": 0.1774960380348653,
      "50": 0.3264659270998415,
      "100": 0.4675118858954041
    },
    "history_3_plus": {
      "queries": 1719,
      "5": 0.06050029086678301,
      "10": 0.10413030831879,
      "20": 0.16579406631762653,
      "50": 0.3054101221640489,
      "100": 0.46713205351948806
    }
  },
  "target_availability": 0.9695325542570952
}


![User history distribution](candidate_k_history_distribution.png)

![Candidate recall curve](candidate_k_recall_curve.png)

## Takeaways

- K=100 covers 13.4% of the 748-item catalog and the median query actually receives 100 candidates. It is useful as an internal retrieval ceiling, but weak as a headline product metric.
- RRF recall rises from 10.9% at K=10 to 16.9% at K=20, 31.1% at K=50, and 46.7% at K=100. The gain is real, but it comes from showing the evaluator a much larger fraction of the catalog.
- LambdaMART improves Recall@10 from 10.94% to 11.57%, a modest +0.64 percentage-point gain.
- Users with at least five interactions are 19.3% of all observed users but 56.2% of the benchmark-eligible users. The benchmark therefore describes the active minority, not the full user base.
- Use candidate Recall@10/20/50 and latency as primary retrieval diagnostics; retain Recall@100 only as a ceiling. Use Recall/NDCG/MRR@5/10 for final ranking. Run a 20/50/100 candidate-budget ablation while holding final K=10 fixed.